## minGPT

Here we start getting into minGPT. We will perform basic setup and run a few experiments. 

We formulate a hypothesis:
Laplace estimation for binary IID data approaches entropy as it sees more of a sequence. We hypothesize that this laplace estimation can be done offline by training the transformer to estimate "true p" that a laplace estimation scheme would produce. This offline learning can therefore produce gains for small n's, even for IID data that has no "structure to exploit"


In [7]:
import sys
sys.path.append('../../')

from mingpt.model import GPT
from mingpt.trainer import Trainer
import torch
from torch.utils.data import Dataset

In [8]:
# Universal variables

seq_samples = 500
num_samples = 10000
iters = 1000
learn_rate = 1e-4
model_save_folder = './models/'

In [9]:

# We generate our dataset

class ModelDataset(Dataset):
    """
    IID binary sequences 
    """

    def __init__(self, p, n=10, num_samples=10000):
        self.n = n
        self.num_samples = num_samples 
        self.p = p

    def _sample(self):
        # sample probability p from uniform (0, 1)
        if callable(self.p):
            p = self.p()
        elif isinstance(self.p, list):
            p = self.p[torch.randint(len(self.p), (1,)).item()]
        else:
            p = self.p
        # generate IID Bernoulli sequence of length self.n
        seq = torch.bernoulli(torch.full((self.n,), p)).long()

        x = seq[:-1].clone()
        y = seq[1:].clone()

        return x, y

    def get_block_size(self):
        return self.n - 1

    def __getitem__(self, idx):
        return self._sample() 
    
    def __len__(self):
        return self.num_samples 


class FixedModelDataset(Dataset):
    def __init__(self, p, n=10, num_samples=10000):
        self.n = n
        self.block_size = n - 1
        # pre-generate all sequences at init
        self.data = []
        for _ in range(num_samples):
            seq = torch.bernoulli(torch.full((n,), p)).long()
            x = seq[:-1].clone()
            y = seq[1:].clone()
            self.data.append((x, y))

    def get_block_size(self):
        return self.block_size

    def __getitem__(self, idx):
        return self.data[idx]  # returns fixed pre-generated sequence

    def __len__(self):
        return len(self.data)
        



In [10]:
import torch.nn.functional as F
import os

def model_configure(dataset, iters=1000, learn_rate=1e-4):
    model_config = GPT.get_default_config()
    model_config.model_type = 'gpt-nano'
    model_config.vocab_size = 2
    model_config.block_size = dataset.get_block_size()
    model = GPT(model_config)
    device = 'mps'
    model = model.to(device)
    train_config = Trainer.get_default_config()
    train_config.learning_rate = learn_rate
    train_config.max_iters = iters
    train_config.num_workers = 0
    train_config.device = 'mps'
    trainer = Trainer(train_config, model, dataset)
    return model, trainer

def batch_end_callback(trainer):
    if trainer.iter_num % 100 == 0:
        print(f"iter {trainer.iter_num}: train loss {trainer.loss.item():.5f}")

def train_run(model, trainer, output_name):
    save_dir = model_save_folder + output_name
    print(f'Saving to {save_dir}')
    trainer.set_callback('on_batch_end', batch_end_callback)
    trainer.run()
    torch.save({'state_dict': model.state_dict(), 'block_size': model.block_size}, save_dir)
    print(f'Saved to {save_dir}')

def load_model(model_dir):
    checkpoint = torch.load(model_dir, map_location='mps')
    model_config = GPT.get_default_config()
    model_config.model_type = 'gpt-nano'
    model_config.vocab_size = 2
    model_config.block_size = checkpoint['block_size']
    model = GPT(model_config)
    model.load_state_dict(checkpoint['state_dict'])
    return model.to('mps').eval()

def train_or_load(model_name, dataset, iters, learn_rate):
    path = model_save_folder + model_name
    if os.path.exists(path):
        print(f'Checkpoint found, loading {model_name}')
        return load_model(path)
    model, trainer = model_configure(dataset, iters=iters, learn_rate=learn_rate)
    train_run(model, trainer, model_name)
    return model


# Model for IID

In [11]:
p_iid_dataset = FixedModelDataset(p=0.3, n=seq_samples, num_samples=num_samples)
model_iid = train_or_load('iid_model.pt', p_iid_dataset, iters=iters, learn_rate=learn_rate)


Checkpoint found, loading iid_model.pt
number of parameters: 0.11M


In [12]:
# iid_model.pt trained above


# Model for p=0.3

In [13]:
p_03_dataset = FixedModelDataset(p=0.3, n=seq_samples, num_samples=num_samples)
model_p03 = train_or_load('p03_model.pt', p_03_dataset, iters=iters, learn_rate=learn_rate)


Checkpoint found, loading p03_model.pt
number of parameters: 0.11M


In [14]:
print(trainer_p03.config.batch_size)

NameError: name 'trainer_p03' is not defined

In [ ]:
# p03_model.pt trained above


# Model for p=0.7

In [ ]:
p_07_dataset = FixedModelDataset(p=0.7, n=seq_samples, num_samples=num_samples)
model_p07 = train_or_load('p07_model.pt', p_07_dataset, iters=iters, learn_rate=learn_rate)


In [ ]:
# p07_model.pt trained above


# Model for p = [0.1, 0.3, 0.5, 0.7, 0.9]

In [ ]:
p_points_dataset = FixedModelDataset(p=[0.1, 0.3, 0.5, 0.7, 0.9], n=seq_samples, num_samples=num_samples)
model_points = train_or_load('points_model.pt', p_points_dataset, iters=iters, learn_rate=learn_rate)


In [ ]:
# points_model.pt trained above
